## OATP prediction

In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, make_scorer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score
import joblib

# Load Assay dataset
data = pd.read_csv("../data/OATP.csv")

# Define a function to convert SMILES to ECFP4 molecular fingerprints
def smiles_to_ecfp4(smiles, radius=2, n_bits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    else:
        return None
    
# Apply fingerprint conversion to the SMILES column
fingerprints = data['SMILES'].apply(smiles_to_ecfp4)

# Remove invalid SMILES entries
valid_indices = fingerprints.notnull()
fingerprints = fingerprints[valid_indices]
labels = data['Assay Response'][valid_indices]

# Convert fingerprints to NumPy array
X = np.array([np.array(fp) for fp in fingerprints])
y = labels.values

# Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Initialize the Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, min_samples_leaf=2)

# Use 5-fold cross-validation to calculate AUC, Accuracy, and F1 Score
cv_auc = cross_val_score(clf, X_resampled, y_resampled, cv=5, scoring=make_scorer(roc_auc_score))
cv_accuracy = cross_val_score(clf, X_resampled, y_resampled, cv=5, scoring=make_scorer(accuracy_score))
cv_f1 = cross_val_score(clf, X_resampled, y_resampled, cv=5, scoring=make_scorer(f1_score))

print("Cross-validated AUC:", cv_auc.mean())
print("Cross-validated Accuracy:", cv_accuracy.mean())
print("Cross-validated F1 Score:", cv_f1.mean())

# Train the model on the resampled dataset
clf.fit(X_resampled, y_resampled)

# Save the trained model locally
model_path = "OATP_random_forest_model.joblib"
joblib.dump(clf, model_path)
print(f"Model saved to {model_path}")

Cross-validated AUC: 0.874092741935484
Cross-validated Accuracy: 0.8741567460317461
Cross-validated F1 Score: 0.8578469079939668
Model saved to OATP_random_forest_model.joblib


#### Predict the OATP target probability of BCF dataset

In [2]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, make_scorer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score
import joblib

# Path to the trained model saved locally
model_path = "OATP_random_forest_model.joblib"

# Define a function to convert SMILES to ECFP4 molecular fingerprints
def smiles_to_ecfp4(smiles, radius=2, n_bits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    else:
        return None

# Load external test dataset and make predictions
def predict_external_test(file_path):
    # Read the external test dataset
    external_data = pd.read_csv(file_path)
    
    # Generate fingerprints from SMILES
    external_fingerprints = external_data['SMILES'].apply(smiles_to_ecfp4)

    # Remove invalid SMILES entries
    valid_indices = external_fingerprints.notnull()
    external_fingerprints = external_fingerprints[valid_indices]
    
    # Convert the fingerprints to a NumPy array
    X_external = np.array([np.array(fp) for fp in external_fingerprints])

    # Load the trained model
    clf_loaded = joblib.load(model_path)

    # Predict probability scores
    external_pred_proba = clf_loaded.predict_proba(X_external)[:, 1]
    
    # Keep only the valid entries
    external_data = external_data[valid_indices]
    external_data['prediction_probability'] = external_pred_proba

    # Print the prediction results
    print(external_data[['SMILES', 'prediction_probability']])
    return external_data

# Example: Use external test set for prediction
external_test_path = "../data/BCF_model_assay_dataset.csv"
predicted_data = predict_external_test(external_test_path)

# Save the prediction results to a CSV file
predicted_data.to_csv("../data/predicted_OATP_results.csv", index=False)

                                                 SMILES  \
0     CCN(CC)c1ccc2c(-c3ccccc3C(=O)O)c3ccc(N(CC)CC)c...   
1                                O=C(O)c1ccc(Cl)c(Cl)c1   
2               Brc1cc(Br)c(-c2c(Br)cc(Br)cc2Br)c(Br)c1   
3                CC(C)Oc1cccc(NC(=O)c2ccccc2C(F)(F)F)c1   
4                                     C1=CCCC=CCCC=CCC1   
...                                                 ...   
1667  CC(C)c1ccc2c(c1)CC[C@H]1[C@](C)(C(=O)O)CCC[C@]21C   
1668                              CC12CCC(CC1)C(C)(C)O2   
1669        CC(Cl)(CCl)OP(=O)(OC(C)(Cl)CCl)OC(C)(Cl)CCl   
1670                                   CCCCCCCCC(Br)CBr   
1671                               CC1CC(C)CC(C)CC(C)C1   

      prediction_probability  
0                   0.241985  
1                   0.240949  
2                   0.694037  
3                   0.236108  
4                   0.794128  
...                      ...  
1667                0.184687  
1668                0.593083  
1669        

## FABP prediction

In [3]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
import joblib

# Load Assay dataset
data = pd.read_csv("../data/FABP.csv")

# Define a function to convert SMILES to ECFP4 molecular fingerprints
def smiles_to_ecfp4(smiles, radius=2, n_bits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    else:
        return None

# Apply fingerprint conversion to the SMILES column
fingerprints = data['SMILES'].apply(smiles_to_ecfp4)

# Remove invalid SMILES entries
valid_indices = fingerprints.notnull()
fingerprints = fingerprints[valid_indices]
labels = data['Assay Response'][valid_indices]

# Convert fingerprints to NumPy array
X = np.array([np.array(fp) for fp in fingerprints])
y = labels.values

# Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Initialize the Random Forest classifier
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
clf.fit(X_train, y_train)

# Save the trained model locally
model_path = "FABP_random_forest_model.joblib"
joblib.dump(clf, model_path)
print(f"Model saved to {model_path}")

# Calculate AUC, Accuracy, and F1 Score
y_pred_proba = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)
auc_score = roc_auc_score(y_test, y_pred_proba)
print("Test AUC Score:", auc_score)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)
f1 = f1_score(y_test, y_pred)
print("Test F1 Score:", f1)

Model saved to FABP_random_forest_model.joblib
Test AUC Score: 0.8125
Test Accuracy: 0.75
Test F1 Score: 0.75


#### Predict the FABP target probability of BCF dataset

In [4]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
import joblib

# Path to the trained model saved locally
model_path = "FABP_random_forest_model.joblib"

# Define a function to convert SMILES strings to ECFP4 molecular fingerprints
def smiles_to_ecfp4(smiles, radius=2, n_bits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    else:
        return None

# Load external test dataset and make predictions
def predict_external_test(file_path):
    # Read the external test dataset
    external_data = pd.read_csv(file_path)
    
    # Generate fingerprints from SMILES
    external_fingerprints = external_data['SMILES'].apply(smiles_to_ecfp4)
    
    # Remove invalid SMILES entries
    valid_indices = external_fingerprints.notnull()
    external_fingerprints = external_fingerprints[valid_indices]
    
    # Convert the fingerprints to a NumPy array
    X_external = np.array([np.array(fp) for fp in external_fingerprints])
    
    # Load the trained model
    clf_loaded = joblib.load(model_path)
    
    # Predict probability scores
    external_pred_proba = clf_loaded.predict_proba(X_external)[:, 1]
    
    # Keep only the valid entries
    external_data = external_data[valid_indices]
    external_data['prediction_probability'] = external_pred_proba
    
    # Print the prediction results
    print(external_data[['SMILES', 'prediction_probability']])
    return external_data

# Example: Use external test set for prediction
external_test_path = "../data/BCF_model_assay_dataset.csv"
predicted_data = predict_external_test(external_test_path)

# Save the prediction results to a CSV file
predicted_data.to_csv("../data/predicted_FABP_results.csv", index=False)

                                                 SMILES  \
0     CCN(CC)c1ccc2c(-c3ccccc3C(=O)O)c3ccc(N(CC)CC)c...   
1                                O=C(O)c1ccc(Cl)c(Cl)c1   
2               Brc1cc(Br)c(-c2c(Br)cc(Br)cc2Br)c(Br)c1   
3                CC(C)Oc1cccc(NC(=O)c2ccccc2C(F)(F)F)c1   
4                                     C1=CCCC=CCCC=CCC1   
...                                                 ...   
1667  CC(C)c1ccc2c(c1)CC[C@H]1[C@](C)(C(=O)O)CCC[C@]21C   
1668                              CC12CCC(CC1)C(C)(C)O2   
1669        CC(Cl)(CCl)OP(=O)(OC(C)(Cl)CCl)OC(C)(Cl)CCl   
1670                                   CCCCCCCCC(Br)CBr   
1671                               CC1CC(C)CC(C)CC(C)C1   

      prediction_probability  
0                      0.215  
1                      0.230  
2                      0.255  
3                      0.220  
4                      0.280  
...                      ...  
1667                   0.365  
1668                   0.425  
1669        